# TARGET 5.1 - Warehouse Assignment Prediction (Which Warehouse Fulfills This Order)

**Goal:** Given an order, predict `fulfilling_warehouse_id` — which warehouse should fulfill it.

This is one half of the original Target 5 notebook, split out so the warehouse pipeline (32 classes, strong signal) isn't bottlenecked by the harder store-assignment problem (200 classes). See `target_5.2_store.ipynb` for the store-assignment counterpart.

## 1. Libraries and data loading

In [4]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import (
    mean_absolute_error, mean_squared_error, r2_score,
    accuracy_score, f1_score, classification_report
)
from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier, ExtraTreesClassifier
from xgboost import XGBClassifier
import joblib
pd.set_option('display.max_columns', 50)
pd.set_option('display.width', 200)

orders     = pd.read_parquet('orders.parquet')
warehouse  = pd.read_parquet('warehouse.parquet')
stores     = pd.read_parquet('stores.parquet')
inventory  = pd.read_parquet('inventory.parquet')
deliveries = pd.read_parquet('deliveries.parquet')
holidays   = pd.read_parquet('holidays.parquet')

print('orders (augmented) :', orders.shape)
print('warehouse          :', warehouse.shape)
print('stores             :', stores.shape)
print('inventory          :', inventory.shape)
print('deliveries         :', deliveries.shape)
print('holidays           :', holidays.shape)


orders (augmented) : (110000, 8)
warehouse          : (32, 6)
stores             : (200, 12)
inventory          : (6000, 4)
deliveries         : (50000, 11)
holidays           : (135, 3)


In [5]:
orders.head(3)

,order_id,region,item_count,created_at,delivery_type,shipment_id,fulfilling_warehouse_id,destination_store_id
0,OR951054,Absheron,9,2026-06-07 06:33:53,express,SH60813,WH_ABSHERON_08,ST0002
1,OR962024,Absheron,2,2022-11-01 03:41:42,daily_product,SH68733,WH_ABSHERON_03,ST0012
2,OR982953,Khachmaz,5,2021-06-19 09:03:08,standard,UNASSIGNED,WH_KHACHMAZ_19,ST0165


## 2. Schema check

In [6]:
print('Missing fulfilling_warehouse_id:', orders['fulfilling_warehouse_id'].isna().sum())
print()
print('Warehouse target — number of classes:', orders['fulfilling_warehouse_id'].nunique())
print()
print('deliveries.order_id overlap with orders.order_id:', deliveries['order_id'].isin(orders['order_id']).mean())


Missing fulfilling_warehouse_id: 0

Warehouse target — number of classes: 32

deliveries.order_id overlap with orders.order_id: 1.0


## 3. Feature engineering — base order/time features

All features below are known at order-creation time (before any warehouse assignment happens), so none of them leak the target.

In [7]:
df = orders.copy()
df['created_at'] = pd.to_datetime(df['created_at'])
df = df.sort_values('created_at').reset_index(drop=True)   # sort ONCE, up front, so every later
                                                             # merge/aggregation stays aligned with df's index

df['order_hour']       = df['created_at'].dt.hour
df['order_dow']        = df['created_at'].dt.dayofweek       # 0 = Monday
df['order_month']      = df['created_at'].dt.month
df['order_is_weekend'] = df['order_dow'].isin([5, 6]).astype(int)

# cyclical encoding (23h and 0h are adjacent, not far apart -- raw integers don't capture that)
df['order_hour_sin'] = np.sin(2 * np.pi * df['order_hour'] / 24)
df['order_hour_cos'] = np.cos(2 * np.pi * df['order_hour'] / 24)
df['dow_sin'] = np.sin(2 * np.pi * df['order_dow'] / 7)
df['dow_cos'] = np.cos(2 * np.pi * df['order_dow'] / 7)

df['item_count_bin'] = pd.cut(
    df['item_count'], bins=[0, 3, 7, 15, np.inf], labels=['small', 'medium', 'large', 'bulk']
)

# tertile / hour-bucket features mirror the SAME rule used to assign the labels
# (region-level item_count tertile, and a 4-way hour bucket) -- these are legitimate,
# leakage-free order-time features, not a shortcut around the target.
df['item_count_tertile'] = df.groupby('region')['item_count'].transform(
    lambda s: pd.qcut(s.rank(method='first'), 3, labels=['low', 'med', 'high'])
)

def hour_bucket(h):
    if 5 <= h < 11: return 'morning'
    if 11 <= h < 16: return 'midday'
    if 16 <= h < 21: return 'evening'
    return 'night'

df['order_hour_bucket'] = df['order_hour'].apply(hour_bucket)

holidays['date'] = pd.to_datetime(holidays['date']).dt.date
df['order_date'] = df['created_at'].dt.date
df['is_holiday'] = df['order_date'].isin(holidays['date']).astype(int)

print('Base features added. df shape:', df.shape)
df[['region', 'item_count', 'item_count_tertile', 'delivery_type', 'order_hour_bucket', 'is_holiday']].head()


Base features added. df shape: (110000, 21)


,region,item_count,item_count_tertile,delivery_type,order_hour_bucket,is_holiday
0,Ganja,10,high,express,night,1
1,Absheron,14,high,standard,night,1
2,Sheki,7,high,express,night,1
3,Khachmaz,3,low,express,night,1
4,Absheron,9,high,express,night,1


## 4. TARGET label-encoding

In [8]:
le_wh = LabelEncoder()
df['warehouse_enc'] = le_wh.fit_transform(df['fulfilling_warehouse_id'])

print(f'Warehouse classes: {len(le_wh.classes_)}')


Warehouse classes: 32


## 5. Static region-level context features (warehouse, inventory, store density)

These come from **static reference tables** — no timestamp, no dependency on the target — so it's safe to merge them onto the full dataset before the train/test split.

In [9]:
wh_region = warehouse.groupby('region').agg(
    wh_count=('warehouse_id', 'count'),
    wh_total_capacity=('capacity', 'sum'),
    wh_total_current_load=('current_load', 'sum'),
    wh_total_inbound_orders=('inbound_orders', 'sum'),
    wh_total_outbound_orders=('outbound_orders', 'sum'),
).reset_index()
wh_region['wh_avg_utilization_pct'] = (wh_region['wh_total_current_load'] / wh_region['wh_total_capacity']).round(4)

inv = inventory.copy()
inv['region'] = inv['warehouse_id'].str.extract(r'^WH_([A-Z]+)_')[0].str.title()
region_fix = {r.upper().replace('_', ''): r for r in warehouse['region'].unique()}
inv['region'] = inv['region'].apply(lambda r: region_fix.get(r.upper(), r) if pd.notna(r) else r)

inv_region = inv.groupby('region').agg(
    inv_total_stock=('stock_level', 'sum'),
    inv_avg_stock=('stock_level', 'mean'),
    inv_low_stock_sku_count=('stock_level', lambda s: (s < 50).sum()),
).reset_index()

store_region = stores.groupby('region').agg(store_count=('store_id', 'count')).reset_index()

df = df.merge(wh_region, on='region', how='left')
df = df.merge(inv_region, on='region', how='left')
df = df.merge(store_region, on='region', how='left')

static_region_cols = [
    'wh_count', 'wh_total_capacity', 'wh_avg_utilization_pct',
    'wh_total_inbound_orders', 'wh_total_outbound_orders',
    'inv_total_stock', 'inv_avg_stock', 'inv_low_stock_sku_count', 'store_count',
]
df[static_region_cols] = df[static_region_cols].fillna(0)

print('Static region-level features added:', static_region_cols)
df[['region'] + static_region_cols].drop_duplicates().reset_index(drop=True)


Static region-level features added: ['wh_count', 'wh_total_capacity', 'wh_avg_utilization_pct', 'wh_total_inbound_orders', 'wh_total_outbound_orders', 'inv_total_stock', 'inv_avg_stock', 'inv_low_stock_sku_count', 'store_count']


,region,wh_count,wh_total_capacity,wh_avg_utilization_pct,wh_total_inbound_orders,wh_total_outbound_orders,inv_total_stock,inv_avg_stock,inv_low_stock_sku_count,store_count
0,Ganja,5,9999.7,0.5920,265,269,186340,174.149533,143,33
1,Absheron,11,20735.2,0.5349,779,851,267495,175.291612,174,59
2,Sheki,2,5950.0,0.6131,136,118,97937,172.424296,69,14
3,Khachmaz,2,1816.2,0.8597,114,100,100219,173.389273,74,13
4,Yevlakh,1,1415.4,0.5362,46,47,70861,178.491184,52,10
5,Nakhchivan,5,12396.3,0.5435,206,227,65195,168.028351,41,41
6,Lankaran,2,4770.1,0.5990,99,90,128191,173.465494,80,17
7,Khankendi,1,2882.4,0.2996,43,38,30857,163.264550,35,1
8,Qazakh,2,4728.0,0.6445,95,98,65431,171.285340,47,7
9,Kalbajar,1,1558.5,0.6136,33,46,29524,181.128834,19,5


## 6. Train / Test split — by TIME, not randomly

Same convention as Targets 3 and 4: sort by `created_at` (already done in Section 3), cut at the 85th percentile.

In [10]:
cutoff_t5 = df['created_at'].quantile(0.85)
train_mask = df['created_at'] < cutoff_t5

print(f'Cutoff date: {cutoff_t5.date()}')
print(f'Train: {train_mask.sum()}   Test: {(~train_mask).sum()}')
print(f'Train period: {df.loc[train_mask, "created_at"].min().date()} -> {df.loc[train_mask, "created_at"].max().date()}')
print(f'Test period:  {df.loc[~train_mask, "created_at"].min().date()} -> {df.loc[~train_mask, "created_at"].max().date()}')


Cutoff date: 2025-06-22
Train: 93500   Test: 16500
Train period: 2020-01-01 -> 2025-06-22
Test period:  2025-06-22 -> 2026-06-12


**Leakage verification test** — every test-set order must have been created strictly after every train-set order.

In [11]:
assert df.loc[train_mask, 'created_at'].max() < df.loc[~train_mask, 'created_at'].min(), \
    "LEAKAGE: a train-set order was created after a test-set order!"

print("LEAKAGE TEST: PASS")
print(f"  Last train order created : {df.loc[train_mask, 'created_at'].max()}")
print(f"  First test order created : {df.loc[~train_mask, 'created_at'].min()}")
print(f"  Gap: {(df.loc[~train_mask, 'created_at'].min() - df.loc[train_mask, 'created_at'].max())}")


LEAKAGE TEST: PASS
  Last train order created : 2025-06-22 16:16:51
  First test order created : 2025-06-22 16:43:28
  Gap: 0 days 00:26:37


## 7. TRAIN-ONLY historical priors (most-common warehouse)

`hist_top_warehouse` mirrors the actual assignment rule used to build the labels — computed **only on the train split**, then mapped onto both train and test.

In [12]:
train_df = df[train_mask]

# --- warehouse prior: (region, item_count_tertile, delivery_type) -> most common warehouse ---
wh_hist = (train_df.groupby(['region', 'item_count_tertile', 'delivery_type', 'fulfilling_warehouse_id'])
           .size().reset_index(name='count'))
wh_totals = train_df.groupby(['region', 'item_count_tertile', 'delivery_type']).size().reset_index(name='total')
wh_hist = wh_hist.merge(wh_totals, on=['region', 'item_count_tertile', 'delivery_type'])
wh_hist['share'] = wh_hist['count'] / wh_hist['total']
wh_top = (wh_hist.sort_values('share', ascending=False)
          .drop_duplicates(['region', 'item_count_tertile', 'delivery_type'])
          [['region', 'item_count_tertile', 'delivery_type', 'fulfilling_warehouse_id', 'share']]
          .rename(columns={'fulfilling_warehouse_id': 'hist_top_warehouse', 'share': 'hist_top_warehouse_share'}))

df = df.merge(wh_top, on=['region', 'item_count_tertile', 'delivery_type'], how='left')

df['hist_top_warehouse'] = df['hist_top_warehouse'].fillna('UNKNOWN')
df['hist_top_warehouse_share'] = df['hist_top_warehouse_share'].fillna(1.0 / max(wh_region['wh_count'].sum(), 1))

print('Historical warehouse prior added.')
df[['region', 'item_count_tertile', 'delivery_type', 'hist_top_warehouse', 'hist_top_warehouse_share']].drop_duplicates().head()


Historical warehouse prior added.


,region,item_count_tertile,delivery_type,hist_top_warehouse,hist_top_warehouse_share
0,Ganja,high,express,WH_GANJA_14,0.877119
1,Absheron,high,standard,WH_ABSHERON_07,0.835664
2,Sheki,high,express,WH_SHEKI_31,0.883090
3,Khachmaz,low,express,WH_KHACHMAZ_19,0.919214
4,Absheron,high,express,WH_ABSHERON_08,0.854521


## 8. TRAIN-ONLY historical delivery performance (region-level)

A specific order's own `delay_minutes` / `actual_duration` happens **after** warehouse assignment, so it can never be used directly as a feature for that same order (that would be leakage). Instead, delay/duration are aggregated **per region, using only the train split**, and mapped onto both train and test as a static-per-region signal.

In [13]:
orders_deliv = df[['order_id', 'region']].merge(
    deliveries[['order_id', 'delay_minutes', 'actual_duration', 'attempt_number']],
    on='order_id', how='left'
)
train_deliv = orders_deliv[train_mask.values]

region_perf = train_deliv.groupby('region').agg(
    region_hist_avg_delay=('delay_minutes', 'mean'),
    region_hist_avg_duration=('actual_duration', 'mean'),
    region_hist_avg_attempts=('attempt_number', 'mean'),
).reset_index()

fallback = train_deliv[['delay_minutes', 'actual_duration', 'attempt_number']].mean()

df = df.merge(region_perf, on='region', how='left')
for col, fb in zip(
    ['region_hist_avg_delay', 'region_hist_avg_duration', 'region_hist_avg_attempts'],
    [fallback['delay_minutes'], fallback['actual_duration'], fallback['attempt_number']]
):
    df[col] = df[col].fillna(fb)

delivery_perf_cols = ['region_hist_avg_delay', 'region_hist_avg_duration', 'region_hist_avg_attempts']
print('Train-only delivery-performance features added:', delivery_perf_cols)
df[['region'] + delivery_perf_cols].drop_duplicates().reset_index(drop=True)


Train-only delivery-performance features added: ['region_hist_avg_delay', 'region_hist_avg_duration', 'region_hist_avg_attempts']


,region,region_hist_avg_delay,region_hist_avg_duration,region_hist_avg_attempts
0,Ganja,10.004373,64.881894,1.053090
1,Absheron,10.084825,65.025084,1.048627
2,Sheki,9.721643,64.746189,1.054767
3,Khachmaz,9.974775,65.013595,1.051597
4,Yevlakh,10.336823,65.833694,1.070999
5,Nakhchivan,9.693811,64.601622,1.050895
6,Lankaran,9.392030,64.894144,1.047996
7,Khankendi,9.949130,64.898302,1.050058
8,Qazakh,9.587887,64.233763,1.041667
9,Kalbajar,12.643599,65.476471,1.027682


## 9. Final feature matrix

In [26]:
cat_cols = ['region', 'delivery_type', 'item_count_bin', 'item_count_tertile',
            'order_hour_bucket', 'hist_top_warehouse']
df_enc = pd.get_dummies(df, columns=cat_cols, prefix=cat_cols)

numeric_cols = [
    'item_count', 'order_hour_sin', 'order_hour_cos', 'dow_sin', 'dow_cos',
    'order_month', 'order_is_weekend', 'is_holiday',
    'hist_top_warehouse_share',
] + delivery_perf_cols + static_region_cols

onehot_cols = [c for c in df_enc.columns if c.startswith(tuple(f'{c}_' for c in cat_cols))]
feature_cols = numeric_cols + onehot_cols

feature_cols = list(dict.fromkeys(feature_cols))

X = df_enc[feature_cols]
y_wh = df_enc['warehouse_enc']

X_train, X_test = X[train_mask.values], X[~train_mask.values]
y_wh_train, y_wh_test = y_wh[train_mask.values], y_wh[~train_mask.values]

print(f'Total features: {len(feature_cols)}')
print('X_train:', X_train.shape, ' X_test:', X_test.shape)


Total features: 75
X_train: (93500, 75)  X_test: (16500, 75)


## 10. Baseline — majority-class classifier

In [15]:
def majority_baseline(y_train, y_test, label):
    majority = y_train.mode().iloc[0]
    pred_test = np.full(len(y_test), majority)
    acc = accuracy_score(y_test, pred_test)
    f1 = f1_score(y_test, pred_test, average='macro')
    rmse = mean_squared_error(y_test, pred_test) ** 0.5
    mae = mean_absolute_error(y_test, pred_test)
    r2 = r2_score(y_test, pred_test)
    print(f'--- {label} majority-class baseline ---')
    print(f'  Accuracy_test: {acc:.4f}   F1_macro_test: {f1:.4f}')
    print(f'  RMSE_test: {rmse:.3f}   MAE_test: {mae:.3f}   R2_test: {r2:.4f}')
    return {'Accuracy_test': acc, 'F1_macro_test': f1, 'RMSE_test': rmse, 'MAE_test': mae, 'R2_test': r2}

wh_baseline = majority_baseline(y_wh_train, y_wh_test, 'WAREHOUSE')


--- WAREHOUSE majority-class baseline ---
  Accuracy_test: 0.0969   F1_macro_test: 0.0055
  RMSE_test: 16.606   MAE_test: 13.258   R2_test: -1.7584


## 11. Comparing 6 models — WAREHOUSE target (`fulfilling_warehouse_id`)

Nominal multi-class target -> Accuracy / F1-macro are the metrics that matter; RMSE/MAE/R2 (on the encoded class index) are reported only for format consistency with Targets 1-4.

In [19]:
def compare_models(X_train, X_test, y_train, y_test):

    X_train = X_train.loc[:, ~X_train.columns.duplicated()].copy()
    X_test = X_test.loc[:, ~X_test.columns.duplicated()].copy()
    X_test = X_test.reindex(columns=X_train.columns, fill_value=0)

    y_train = pd.Series(np.asarray(y_train).ravel(), index=X_train.index if len(y_train) == len(X_train) else None)
    y_test = pd.Series(np.asarray(y_test).ravel(), index=X_test.index if len(y_test) == len(X_test) else None)

    candidate_models = {
        'LogisticRegression': LogisticRegression(max_iter=1000, random_state=42),
        'KNN': KNeighborsClassifier(n_neighbors=15),
        'RandomForest': RandomForestClassifier(random_state=42, n_jobs=-1),
        'GradientBoosting': GradientBoostingClassifier(random_state=42),
        'ExtraTrees': ExtraTreesClassifier(random_state=42, n_jobs=-1),
        'XGBoost': XGBClassifier(random_state=42, eval_metric='mlogloss'),
    }
    results = []
    fitted_models = {}
    for name, model in candidate_models.items():
        model.fit(X_train, y_train)
        fitted_models[name] = model
        pred_train = model.predict(X_train)
        pred_test = model.predict(X_test)
        results.append({
            'Model': name,
            'Accuracy_train': accuracy_score(y_train, pred_train),
            'Accuracy_test': accuracy_score(y_test, pred_test),
            'F1_macro_train': f1_score(y_train, pred_train, average='macro'),
            'F1_macro_test': f1_score(y_test, pred_test, average='macro'),
            'RMSE_train': mean_squared_error(y_train, pred_train) ** 0.5,
            'RMSE_test': mean_squared_error(y_test, pred_test) ** 0.5,
            'MAE_train': mean_absolute_error(y_train, pred_train),
            'MAE_test': mean_absolute_error(y_test, pred_test),
            'R2_train': r2_score(y_train, pred_train),
            'R2_test': r2_score(y_test, pred_test),
        })
    comparison_df = pd.DataFrame(results).sort_values('F1_macro_test', ascending=False).reset_index(drop=True)
    return comparison_df, fitted_models

wh_comparison_df, wh_fitted_models = compare_models(X_train, X_test, y_wh_train, y_wh_test)
wh_comparison_df

c:\Users\Phaic\anaconda3\Lib\site-packages\sklearn\linear_model\_logistic.py:599: ConvergenceWarning: lbfgs failed to converge after 1000 iteration(s) (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT

Increase the number of iterations to improve the convergence (max_iter=1000).
You might also want to scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(


,Model,Accuracy_train,Accuracy_test,F1_macro_train,F1_macro_test,RMSE_train,RMSE_test,MAE_train,MAE_test,R2_train,R2_test
0,GradientBoosting,0.868813,0.861818,0.814235,0.805600,1.269662,1.283933,0.370888,0.387879,0.983508,0.983510
1,KNN,0.853904,0.858788,0.799957,0.802374,1.314375,1.285938,0.407455,0.391455,0.982326,0.983458
2,XGBoost,0.886503,0.857515,0.838423,0.799976,1.243146,1.286456,0.342909,0.392788,0.984190,0.983445
3,RandomForest,0.955348,0.827636,0.946327,0.777449,0.922349,1.484587,0.161701,0.499515,0.991297,0.977953
4,ExtraTrees,0.955422,0.808667,0.945616,0.758067,0.926214,1.518332,0.162364,0.539636,0.991224,0.976939
5,LogisticRegression,0.347390,0.352424,0.191988,0.191297,3.493242,3.439071,2.277540,2.226606,0.875164,0.881690


In [22]:
X_train = X_train.loc[:, ~X_train.columns.duplicated()].copy()
X_test = X_test.loc[:, ~X_test.columns.duplicated()].copy()
X_test = X_test.reindex(columns=X_train.columns, fill_value=0)

print(X_train.shape, X_test.shape)
print("Any dupes left?", X_train.columns.duplicated().any())

(93500, 75) (16500, 75)
Any dupes left? False


**Best warehouse model** — selected by **Test F1-macro** (the metric that actually reflects classification quality for a nominal target), with a Train/Test gap cap to control overfitting.

In [20]:
MAX_ACCEPTABLE_GAP = 0.15

wh_comparison_df['gap'] = wh_comparison_df['F1_macro_train'] - wh_comparison_df['F1_macro_test']
eligible = wh_comparison_df[wh_comparison_df['gap'] <= MAX_ACCEPTABLE_GAP]
if eligible.empty:
    print("No model meets the gap threshold — falling back to best test F1 overall.")
    eligible = wh_comparison_df

best_wh_model_name = eligible.sort_values('F1_macro_test', ascending=False).iloc[0]['Model']
best_wh_row = wh_comparison_df[wh_comparison_df['Model'] == best_wh_model_name].iloc[0]
best_wh_model = wh_fitted_models[best_wh_model_name]

print(f"Best warehouse model: {best_wh_model_name}")
print(f"  Test Accuracy : {best_wh_row['Accuracy_test']:.4f}")
print(f"  Test F1-macro : {best_wh_row['F1_macro_test']:.4f}")
print(f"  Test RMSE     : {best_wh_row['RMSE_test']:.3f}")
print(f"  Test MAE      : {best_wh_row['MAE_test']:.3f}")
print(f"  Test R2       : {best_wh_row['R2_test']:.4f}")
print(f"  Train/Test F1 gap : {best_wh_row['gap']:.4f}")


Best warehouse model: GradientBoosting
  Test Accuracy : 0.8618
  Test F1-macro : 0.8056
  Test RMSE     : 1.284
  Test MAE      : 0.388
  Test R2       : 0.9835
  Train/Test F1 gap : 0.0086


## 12. Hyperparameter tuning

In [23]:
# =========================================================================
# GridSearchCV — WAREHOUSE target — GradientBoostingClassifier (best model)
# =========================================================================
from sklearn.model_selection import GridSearchCV

param_grid_wh = {
    'n_estimators':     [100],    
    'max_depth':        [3],       
    'learning_rate':    [0.05],  
    'subsample':        [0.7],    
    'min_samples_split':[2],      
    'min_samples_leaf': [1],     
}

gb_base = GradientBoostingClassifier(random_state=42)

grid_search_wh = GridSearchCV(
    estimator=gb_base,
    param_grid=param_grid_wh,
    scoring='f1_macro',
    cv=10,              
    n_jobs=-1,
    verbose=2,
    refit=True,
)

grid_search_wh.fit(X_train, y_wh_train)

print(f"Number of parameter combinations tried: {len(grid_search_wh.cv_results_['params'])}")
print(f"Total fits run (combos x folds): {len(grid_search_wh.cv_results_['params']) * 10}")
print(f"Best params: {grid_search_wh.best_params_}")
print(f"Best CV F1-macro score: {grid_search_wh.best_score_:.4f}")

best_wh_model_tuned = grid_search_wh.best_estimator_

pred_train = best_wh_model_tuned.predict(X_train)
pred_test  = best_wh_model_tuned.predict(X_test)

tuned_wh_results = {
    'Model': 'GradientBoosting_tuned',
    'Accuracy_train': accuracy_score(y_wh_train, pred_train),
    'Accuracy_test':  accuracy_score(y_wh_test, pred_test),
    'F1_macro_train': f1_score(y_wh_train, pred_train, average='macro'),
    'F1_macro_test':  f1_score(y_wh_test, pred_test, average='macro'),
    'RMSE_train': mean_squared_error(y_wh_train, pred_train) ** 0.5,
    'RMSE_test':  mean_squared_error(y_wh_test, pred_test) ** 0.5,
    'MAE_train':  mean_absolute_error(y_wh_train, pred_train),
    'MAE_test':   mean_absolute_error(y_wh_test, pred_test),
    'R2_train':   r2_score(y_wh_train, pred_train),
    'R2_test':    r2_score(y_wh_test, pred_test),
}

tuned_wh_df = pd.DataFrame([tuned_wh_results])
tuned_wh_df

Fitting 10 folds for each of 1 candidates, totalling 10 fits
Number of parameter combinations tried: 1
Total fits run (combos x folds): 10
Best params: {'learning_rate': 0.05, 'max_depth': 3, 'min_samples_leaf': 1, 'min_samples_split': 2, 'n_estimators': 100, 'subsample': 0.7}
Best CV F1-macro score: 0.8056


,Model,Accuracy_train,Accuracy_test,F1_macro_train,F1_macro_test,RMSE_train,RMSE_test,MAE_train,MAE_test,R2_train,R2_test
0,GradientBoosting_tuned,0.867818,0.862364,0.812958,0.806595,1.271535,1.282257,0.372717,0.386364,0.98346,0.983553


## 13. Save Best Model with joblib

In [24]:
import os

os.makedirs("models", exist_ok=True)

wh_model_path = f"models/target5_warehouse_{best_wh_model_name}.joblib"
joblib.dump({'model': best_wh_model, 'label_encoder': le_wh, 'feature_cols': feature_cols}, wh_model_path)
print(f"Saved best warehouse model ({best_wh_model_name}) to: {wh_model_path}")


Saved best warehouse model (GradientBoosting) to: models/target5_warehouse_GradientBoosting.joblib


## 14. 7-Day Warehouse Forecast (by Origin Region)

For each region, next week's most likely warehouse is forecast day-by-day. The region's **typical order profile** (mean `item_count`, most common `delivery_type`) is frozen, and only calendar-dependent features vary across the next 7 days.

In [27]:
FORECAST_DAYS = 7
last_date_t5 = df['created_at'].max().normalize()
future_dates_t5 = pd.date_range(last_date_t5 + pd.Timedelta(days=1), periods=FORECAST_DAYS, freq='D')

region_profile = df.groupby('region').agg({
    'item_count': 'mean',
    'delivery_type': lambda x: x.mode().iloc[0],
}).reset_index()

rows = []
meta = []
for _, reg in region_profile.iterrows():
    item_count_val = reg['item_count']
    tertile_lookup = df[df['region'] == reg['region']].groupby('item_count_tertile')['item_count'].mean()
    closest_tertile = (tertile_lookup - item_count_val).abs().idxmin() if len(tertile_lookup) else 'med'

    for fdate in future_dates_t5:
        hb = hour_bucket(12)  # midday, representative default hour for the daily outlook
        row = {c: 0 for c in feature_cols}
        row['item_count'] = item_count_val
        row['order_hour_sin'] = np.sin(2 * np.pi * 12 / 24)
        row['order_hour_cos'] = np.cos(2 * np.pi * 12 / 24)
        row['dow_sin'] = np.sin(2 * np.pi * fdate.dayofweek / 7)
        row['dow_cos'] = np.cos(2 * np.pi * fdate.dayofweek / 7)
        row['order_month'] = fdate.month
        row['order_is_weekend'] = int(fdate.dayofweek >= 5)
        row['is_holiday'] = int(fdate.date() in set(holidays['date']))

        for col_prefix, val in [('region', reg['region']), ('delivery_type', reg['delivery_type']),
                                 ('item_count_tertile', closest_tertile), ('order_hour_bucket', hb)]:
            col = f'{col_prefix}_{val}'
            if col in row:
                row[col] = 1

        # static region context + train-only priors/perf for this region
        region_row = df[df['region'] == reg['region']].iloc[0]
        for col in static_region_cols + delivery_perf_cols:
            row[col] = region_row[col]
        row['hist_top_warehouse_share'] = region_row['hist_top_warehouse_share']
        wh_col = f"hist_top_warehouse_{region_row['hist_top_warehouse']}"
        if wh_col in row:
            row[wh_col] = 1

        rows.append(row)
        meta.append({'region': reg['region'], 'date': fdate.date(), 'day_of_week': fdate.day_name()})

X_batch = pd.DataFrame(rows)[feature_cols]

wh_preds_enc = best_wh_model.predict(X_batch)
wh_preds = le_wh.inverse_transform(wh_preds_enc)

wh_proba = best_wh_model.predict_proba(X_batch).max(axis=1) if hasattr(best_wh_model, 'predict_proba') else np.full(len(wh_preds), np.nan)

forecast_df_t5 = pd.DataFrame(meta)
forecast_df_t5['predicted_warehouse_id'] = wh_preds
forecast_df_t5['warehouse_confidence'] = np.round(wh_proba, 4)

print(f"7-day warehouse outlook ({future_dates_t5[0].date()} to {future_dates_t5[-1].date()})")
print(f"  Warehouse model: {best_wh_model_name}")
forecast_df_t5


7-day warehouse outlook (2026-06-13 to 2026-06-19)
  Warehouse model: GradientBoosting


,region,date,day_of_week,predicted_warehouse_id,warehouse_confidence
0,Absheron,2026-06-13,Saturday,WH_ABSHERON_07,0.8156
1,Absheron,2026-06-14,Sunday,WH_ABSHERON_07,0.8199
2,Absheron,2026-06-15,Monday,WH_ABSHERON_07,0.8107
3,Absheron,2026-06-16,Tuesday,WH_ABSHERON_07,0.8256
4,Absheron,2026-06-17,Wednesday,WH_ABSHERON_07,0.7787
...,...,...,...,...,...
65,Yevlakh,2026-06-15,Monday,WH_YEVLAKH_32,1.0000
66,Yevlakh,2026-06-16,Tuesday,WH_YEVLAKH_32,1.0000
67,Yevlakh,2026-06-17,Wednesday,WH_YEVLAKH_32,1.0000
68,Yevlakh,2026-06-18,Thursday,WH_YEVLAKH_32,1.0000


### 7-day warehouse forecast — JSON output

In [28]:
import json

json_records_t5 = []
for region, grp in forecast_df_t5.groupby('region'):
    grp = grp.sort_values('date')
    json_records_t5.append({
        "region": region,
        "warehouse_model": best_wh_model_name,
        "forecast": [
            {
                "date": str(r['date']),
                "day_of_week": r['day_of_week'],
                "predicted_warehouse_id": r['predicted_warehouse_id'],
                "warehouse_confidence": float(r['warehouse_confidence']) if not np.isnan(r['warehouse_confidence']) else None,
            }
            for _, r in grp.iterrows()
        ],
    })

forecast_json_t5 = json.dumps(json_records_t5, indent=2, ensure_ascii=False)

with open("target5_1_7day_warehouse_forecast.json", "w", encoding="utf-8") as f:
    f.write(forecast_json_t5)

print(f"Saved 7-day warehouse forecast for {len(json_records_t5)} regions to target5_1_7day_warehouse_forecast.json")
print(forecast_json_t5[:1200])


Saved 7-day warehouse forecast for 10 regions to target5_1_7day_warehouse_forecast.json
[
  {
    "region": "Absheron",
    "warehouse_model": "GradientBoosting",
    "forecast": [
      {
        "date": "2026-06-13",
        "day_of_week": "Saturday",
        "predicted_warehouse_id": "WH_ABSHERON_07",
        "warehouse_confidence": 0.8156
      },
      {
        "date": "2026-06-14",
        "day_of_week": "Sunday",
        "predicted_warehouse_id": "WH_ABSHERON_07",
        "warehouse_confidence": 0.8199
      },
      {
        "date": "2026-06-15",
        "day_of_week": "Monday",
        "predicted_warehouse_id": "WH_ABSHERON_07",
        "warehouse_confidence": 0.8107
      },
      {
        "date": "2026-06-16",
        "day_of_week": "Tuesday",
        "predicted_warehouse_id": "WH_ABSHERON_07",
        "warehouse_confidence": 0.8256
      },
      {
        "date": "2026-06-17",
        "day_of_week": "Wednesday",
        "predicted_warehouse_id": "WH_ABSHERON_07",
      